<a href="https://colab.research.google.com/github/rulon194-blip/week7-lab/blob/main/Lab_Image_Handling_and_Convolution_A85204924070.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIML301: Deep Learning and Neural Networks
## Lab: Image Handling and Convolution

**Topic:** Applying filters, visualising feature maps, and studying the effect of kernel size, stride and padding.

**Duration:** 2 hours

| Part | What happens | Time |
|---|---|---|
| Session 1 | Guided walkthrough with the instructor | 60 min |
| Session 2 | Graded exercises, done independently (100 marks) | 30 min work + submission |

**By the end of this lab you will be able to:**

1. Explain how a computer stores a grayscale image and a colour image.
2. Convert an image into a PyTorch tensor with the shape `(batch, channels, height, width)`.
3. Explain what convolution does, step by step.
4. Apply blur, sharpen and edge-detection filters to an image.
5. Predict the output size of a convolution from the kernel size, stride and padding.
6. Build a convolution layer and visualise its feature maps.

---
# SESSION 1: Guided Walkthrough
---

## Part 0: Setup

We use four libraries:

- **NumPy** to work with arrays of numbers.
- **Matplotlib** to display images.
- **PyTorch** to perform convolutions (the same tool used to build real neural networks).
- **scikit-image** only to load a few sample images. No internet download is needed.

Run the cell below once. It also defines three small helper functions that we use throughout the lab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from skimage import data

plt.rcParams["figure.dpi"] = 90


def to_tensor(image):
    # Convert a 2-D NumPy image (height, width) with values 0-255
    # into a PyTorch tensor of shape (1, 1, height, width) with values 0-1.
    t = torch.tensor(image / 255.0, dtype=torch.float32)
    return t.reshape(1, 1, image.shape[0], image.shape[1])


def to_numpy(t):
    # Convert a PyTorch tensor back to a NumPy array we can display.
    return t.detach().squeeze().numpy()


def show_images(images, titles, cols=4):
    # Display several images side by side (a few per row).
    if len(images) < cols:
        cols = len(images)
    rows = (len(images) + cols - 1) // cols
    plt.figure(figsize=(4 * cols, 4 * rows))
    for i in range(len(images)):
        img = images[i]
        if torch.is_tensor(img):
            img = to_numpy(img)
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img, cmap="gray")
        plt.title(titles[i])
        plt.axis("off")
    plt.tight_layout()
    plt.show()


print("PyTorch version:", torch.__version__)
print("Setup complete.")

## Part 1: An Image Is a Grid of Numbers

A **grayscale** image is a 2-D table of numbers. Each number is one **pixel**.

- `0` means black.
- `255` means white.
- Numbers in between are shades of grey.

Let us load a sample photograph and look at its size and values.

In [ ]:
gray = data.camera()   # a 512 x 512 grayscale photograph

print("Shape (height, width):", gray.shape)
print("Data type:", gray.dtype)
print("Smallest pixel value:", gray.min())
print("Largest pixel value:", gray.max())

show_images([gray], ["Grayscale image"])

### Zooming in

If we cut out a tiny 8 x 8 piece of the photo, we can see the actual numbers. Notice how bright areas have large numbers and dark areas have small numbers.

In [ ]:
patch = gray[196:204, 188:196]   # rows 196 to 203, columns 188 to 195
print(patch)

plt.figure(figsize=(5, 5))
plt.imshow(patch, cmap="gray")
for row in range(8):
    for col in range(8):
        value = patch[row, col]
        text_color = "black" if value > 128 else "white"
        plt.text(col, row, str(value), ha="center", va="center", color=text_color)
plt.title("8 x 8 patch with pixel values")
plt.axis("off")
plt.show()

### Colour images

A **colour** image has **three** tables stacked together, one for each colour channel: **Red, Green and Blue (RGB)**. So its shape is `(height, width, 3)`.

In [ ]:
color = data.astronaut()   # a 512 x 512 colour photograph

print("Shape (height, width, channels):", color.shape)

red = color[:, :, 0]
green = color[:, :, 1]
blue = color[:, :, 2]

plt.figure(figsize=(16, 4))
plt.subplot(1, 4, 1); plt.imshow(color); plt.title("Colour image"); plt.axis("off")
plt.subplot(1, 4, 2); plt.imshow(red); plt.title("Red channel"); plt.axis("off")
plt.subplot(1, 4, 3); plt.imshow(green); plt.title("Green channel"); plt.axis("off")
plt.subplot(1, 4, 4); plt.imshow(blue); plt.title("Blue channel"); plt.axis("off")
plt.show()

**Observation:** the red suit looks bright in the red channel and dark in the blue channel. Each channel only measures how much of one colour is present.

## Part 2: From an Image to a PyTorch Tensor

PyTorch expects images in a specific shape:

$$\text{(batch, channels, height, width)}$$

- **batch**: how many images we process at once (here, 1).
- **channels**: 1 for grayscale, 3 for colour.
- **height, width**: the image size.

We also divide by 255 so that all pixel values are between 0 and 1. Neural networks train better with small numbers.

In [ ]:
# Step by step
x = torch.tensor(gray / 255.0, dtype=torch.float32)
print("After converting:", x.shape)

x = x.reshape(1, 1, 512, 512)
print("After reshaping:", x.shape, " -> (batch, channels, height, width)")

print("Smallest value:", x.min().item(), " Largest value:", x.max().item())

# The helper function to_tensor() does exactly the same thing in one line
x = to_tensor(gray)
print("Using the helper:", x.shape)

For a colour image, the channels must move to the front: from `(H, W, 3)` to `(1, 3, H, W)`. We use `permute` to reorder the dimensions.

In [ ]:
x_color = torch.tensor(color / 255.0, dtype=torch.float32)
print("Original order (H, W, C):", x_color.shape)

x_color = x_color.permute(2, 0, 1)     # move channels to the front -> (C, H, W)
x_color = x_color.unsqueeze(0)         # add a batch dimension      -> (1, C, H, W)
print("PyTorch order (N, C, H, W):", x_color.shape)

## Part 3: What Is Convolution?

A **kernel** (also called a **filter**) is a small table of numbers, for example 3 x 3.

Convolution works like this:

1. Place the kernel on the top-left corner of the image.
2. Multiply each kernel number by the pixel under it.
3. Add all the results together. This sum is **one** output pixel.
4. Slide the kernel one step to the right and repeat. At the end of a row, move down one step.

The result is a new image called the **output** or **feature map**.

Let us try this on a tiny 6 x 6 image that is dark on the left and bright on the right (a vertical edge).

In [ ]:
tiny = np.array([
    [0, 0, 0, 9, 9, 9],
    [0, 0, 0, 9, 9, 9],
    [0, 0, 0, 9, 9, 9],
    [0, 0, 0, 9, 9, 9],
    [0, 0, 0, 9, 9, 9],
    [0, 0, 0, 9, 9, 9],
], dtype=float)

# A kernel that reacts to vertical edges: negative on the left, positive on the right
kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
], dtype=float)

# Window 1: the top-left 3 x 3 window (a flat dark area)
window = tiny[0:3, 0:3]
print("Window 1:\n", window.astype(int))
print("Window 1 x kernel:\n", (window * kernel).astype(int))
print("Sum =", int(np.sum(window * kernel)))

In [ ]:
# Window 2: move one step to the right, now the window covers the edge
window = tiny[0:3, 1:4]
print("Window 2:\n", window.astype(int))
print("Window 2 x kernel:\n", (window * kernel).astype(int))
print("Sum =", int(np.sum(window * kernel)))

Window 1 is a flat area, so the output is 0: no edge. Window 2 sits on the edge, so the output is large (27). Now let us write a simple function that slides the kernel over the **whole** image.

In [ ]:
def my_conv2d(image, kernel):
    k = kernel.shape[0]                    # kernel size, e.g. 3
    out_h = image.shape[0] - k + 1         # output height
    out_w = image.shape[1] - k + 1         # output width
    output = np.zeros((out_h, out_w))

    for i in range(out_h):                 # move down
        for j in range(out_w):             # move right
            window = image[i:i + k, j:j + k]
            output[i, j] = np.sum(window * kernel)
    return output


result = my_conv2d(tiny, kernel)
print("Input size :", tiny.shape)
print("Output size:", result.shape)
print(result)

**Observation:**

- The output is large (27) only where the window covers the edge, and 0 in flat areas. The kernel has **detected** the edge.
- The output (4 x 4) is **smaller** than the input (6 x 6). A 3 x 3 kernel cannot be centred on the border pixels. We will fix this later with **padding**.

### The same thing in PyTorch

Our loop is slow on large images. PyTorch has a fast built-in function, `F.conv2d`. It needs both the image and the kernel as 4-D tensors. Let us check that it gives the same answer.

(Note: PyTorch does not flip the kernel. Strictly speaking this operation is called *cross-correlation*, but in deep learning everyone calls it convolution.)

In [ ]:
tiny_t = torch.tensor(tiny, dtype=torch.float32).reshape(1, 1, 6, 6)
kernel_t = torch.tensor(kernel, dtype=torch.float32).reshape(1, 1, 3, 3)

result_t = F.conv2d(tiny_t, kernel_t)
print(result_t.squeeze())

Same result. From now on we use a small helper, `apply_filter`, that takes a NumPy kernel and applies it with PyTorch.

In [ ]:
def apply_filter(x, kernel, stride=1, padding=0):
    # x      : image tensor of shape (1, 1, H, W)
    # kernel : 2-D NumPy array, e.g. 3 x 3
    k = torch.tensor(kernel, dtype=torch.float32)
    k = k.reshape(1, 1, k.shape[0], k.shape[1])
    return F.conv2d(x, k, stride=stride, padding=padding)

## Part 4: Classic Filters on a Real Image

Different kernels produce different effects. Here are five well-known ones:

| Filter | What it does | Idea behind the numbers |
|---|---|---|
| Identity | Changes nothing | Only the centre is 1 |
| Box blur | Smooths the image | Averages all 9 neighbours |
| Sharpen | Makes details crisper | Boosts the centre, subtracts neighbours |
| Sobel X | Finds **vertical** edges | Right side minus left side |
| Sobel Y | Finds **horizontal** edges | Bottom minus top |

In [ ]:
identity = np.array([[0, 0, 0],
                     [0, 1, 0],
                     [0, 0, 0]])

box_blur = np.ones((3, 3)) / 9

sharpen = np.array([[ 0, -1,  0],
                    [-1,  5, -1],
                    [ 0, -1,  0]])

sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]])

sobel_y = np.array([[-1, -2, -1],
                    [ 0,  0,  0],
                    [ 1,  2,  1]])

x = to_tensor(gray)

out_identity = apply_filter(x, identity)
out_blur = apply_filter(x, box_blur)
out_sharpen = apply_filter(x, sharpen)
out_sobel_x = apply_filter(x, sobel_x)
out_sobel_y = apply_filter(x, sobel_y)

show_images(
    [out_identity, out_blur, out_sharpen],
    ["Identity", "Box blur 3x3", "Sharpen"],
)

# Edge outputs can be negative, so we show the absolute value (the strength of the edge)
show_images(
    [out_sobel_x.abs(), out_sobel_y.abs()],
    ["Sobel X (vertical edges)", "Sobel Y (horizontal edges)"],
)

**Observation:** Sobel X highlights the vertical lines (the tripod legs, the man's side), while Sobel Y highlights horizontal lines (the horizon, the shoulders). The kernel decides **what pattern** the output responds to.

**Key idea for deep learning:** in a Convolutional Neural Network (CNN), we do **not** choose these numbers by hand. The network **learns** the kernel values during training.

## Part 5: How Big Is the Output?

Three settings control the output size:

- **Kernel size (k):** the width of the filter, e.g. 3 means 3 x 3.
- **Stride (s):** how many pixels the kernel jumps each step.
- **Padding (p):** how many rows/columns of zeros we add around the border.

For an input of size **n**, the output size is:

$$\text{output} = \left\lfloor \frac{n + 2p - k}{s} \right\rfloor + 1$$

The symbol $\lfloor \; \rfloor$ means "round down" (in Python: `//`).

**Worked example:** our image is n = 512, kernel k = 3, stride s = 1, padding p = 0.

$$\text{output} = \frac{512 + 0 - 3}{1} + 1 = 510$$

Let us confirm with the result from Part 4.

In [ ]:
print("Input shape :", x.shape)
print("Output shape:", out_blur.shape)

In the next three parts we change **one** setting at a time and watch what happens.

## Part 6: Effect of Kernel Size

We use box-blur kernels of size 3, 9 and 21. A box blur of size k is simply a k x k table where every value is `1 / (k * k)`.

In [ ]:
images = []
titles = []

for k in [3, 9, 21]:
    kernel = np.ones((k, k)) / (k * k)
    out = apply_filter(x, kernel)
    print(f"kernel {k:2d} x {k:<2d} -> output shape {tuple(out.shape)}")
    images.append(out)
    titles.append(f"k = {k}, output {out.shape[2]} x {out.shape[3]}")

show_images(images, titles)

**Observations:**

1. A **bigger kernel** looks at a bigger area at once, so the image becomes **more blurred**. In a CNN this means each output pixel "sees" a larger region of the input (its **receptive field**).
2. With no padding, a **bigger kernel** gives a **smaller output** (512 - 21 + 1 = 492).
3. A bigger kernel also has more numbers, so it needs more computation. This is why most modern CNNs use small 3 x 3 kernels and stack many layers instead.

## Part 7: Effect of Stride

Stride is the size of the jump. With stride 2, the kernel skips every other position, so the output is about **half** as wide and half as tall.

In [ ]:
images = []
titles = []

for s in [1, 2, 4]:
    out = apply_filter(x, box_blur, stride=s)
    print(f"stride {s} -> output shape {tuple(out.shape)}")
    images.append(out)
    titles.append(f"stride = {s}, output {out.shape[2]} x {out.shape[3]}")

show_images(images, titles)

**Check with the formula** for stride 2: (512 + 0 - 3) // 2 + 1 = 509 // 2 + 1 = 254 + 1 = **255**.

**Observations:**

1. A larger stride **shrinks** the output a lot. Stride 4 leaves only 128 x 128 pixels, so the picture looks more blocky.
2. Stride is a cheap way to **reduce the size** of feature maps inside a CNN, which makes the network faster.
3. The cost: some fine detail is lost.

## Part 8: Effect of Padding

Padding adds a frame of zeros around the image before the convolution. Look at a small example first.

In [ ]:
small = torch.ones(1, 1, 4, 4)
padded = F.pad(small, (1, 1, 1, 1))   # add 1 zero on left, right, top, bottom

print("Before padding:", small.shape)
print(small.squeeze())
print("\nAfter padding = 1:", padded.shape)
print(padded.squeeze())

Now compare padding 0 and padding 1 with a 3 x 3 kernel on our photo.

In [ ]:
out_p0 = apply_filter(x, box_blur, padding=0)
out_p1 = apply_filter(x, box_blur, padding=1)

print("Input              :", tuple(x.shape))
print("k=3, padding = 0 ->", tuple(out_p0.shape))
print("k=3, padding = 1 ->", tuple(out_p1.shape), " (same size as the input)")

**Rule of thumb:** with stride 1, choosing **p = (k - 1) / 2** keeps the output the **same size** as the input. This is often called **"same" padding**.

| Kernel size k | Padding for "same" output |
|---|---|
| 3 | 1 |
| 5 | 2 |
| 7 | 3 |

### Side effect of zero padding

Zeros are black. With a large kernel, the blur near the border mixes in many black pixels, creating a dark frame. Another option is **reflect padding**, which copies the nearby pixels instead of using zeros.

In [ ]:
big_blur = np.ones((21, 21)) / (21 * 21)

out_zero_pad = apply_filter(x, big_blur, padding=10)          # zeros around the border

x_reflect = F.pad(x, (10, 10, 10, 10), mode="reflect")        # copy nearby pixels
out_reflect_pad = apply_filter(x_reflect, big_blur, padding=0)

print("Zero padding output   :", tuple(out_zero_pad.shape))
print("Reflect padding output:", tuple(out_reflect_pad.shape))

show_images(
    [out_zero_pad, out_reflect_pad],
    ["Zero padding (dark border)", "Reflect padding (no dark border)"],
)

### Summary of the three settings

| Setting | If we make it bigger... | Output size | Main use in CNNs |
|---|---|---|---|
| Kernel size | Looks at a larger area, more blur, more computation | Smaller | Choose how much context each pixel sees |
| Stride | Skips more positions | Much smaller (about n / s) | Reduce size, speed up |
| Padding | Adds more border | Larger | Keep the size the same, keep border information |

## Part 9: Convolution Layers in PyTorch

In a real network we do not call `F.conv2d` with our own kernel. We create a **layer** with `nn.Conv2d`:

```python
nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
```

- **in_channels:** channels coming in (1 for grayscale, 3 for colour).
- **out_channels:** how many different filters the layer has. Each filter produces one output image, called a **feature map**.

The layer stores its kernels in `.weight`, with shape `(out_channels, in_channels, k, k)`.

In [ ]:
conv = nn.Conv2d(in_channels=3, out_channels=4, kernel_size=3, stride=1, padding=1)

print("Weight shape:", conv.weight.shape, " -> (out_channels, in_channels, k, k)")

out = conv(x_color)   # the colour astronaut tensor from Part 2
print("Input shape :", x_color.shape)
print("Output shape:", out.shape, " -> 4 feature maps, each 512 x 512")

**Important:** for a colour image, each filter has **3 layers** (one per input channel). It looks at red, green and blue together and adds everything into **one** feature map.

## Part 10: Visualising Feature Maps

A **feature map** is the output of one filter. It shows **where** in the image that filter found its pattern. Bright = strong response.

### 10.1 A layer with filters we choose

We create a layer with 4 filters and put our hand-made kernels into it. After the convolution we apply **ReLU**, which keeps positive values and turns negatives into 0, just as in a real CNN.

In [ ]:
layer = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, padding=1, bias=False)

with torch.no_grad():   # we are setting values by hand, not training
    layer.weight[0, 0] = torch.tensor(sobel_x, dtype=torch.float32)
    layer.weight[1, 0] = torch.tensor(sobel_y, dtype=torch.float32)
    layer.weight[2, 0] = torch.tensor(box_blur, dtype=torch.float32)
    layer.weight[3, 0] = torch.tensor(sharpen, dtype=torch.float32)

feature_maps = torch.relu(layer(x))
print("Feature maps shape:", feature_maps.shape)

maps = [feature_maps[0, i] for i in range(4)]
show_images(maps, ["Map 0: Sobel X", "Map 1: Sobel Y", "Map 2: Blur", "Map 3: Sharpen"])

### 10.2 A layer with random filters

When a CNN is created, its filters start as **random numbers**. Let us see what 8 random 3 x 3 filters and their feature maps look like.

In [ ]:
torch.manual_seed(1)   # so everyone gets the same random numbers
random_layer = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)

# Look at the 8 filters themselves (each one is a 3 x 3 grid)
filters = [random_layer.weight[i, 0] for i in range(8)]
show_images(filters, [f"Filter {i}" for i in range(8)])

# Look at the 8 feature maps they produce
random_maps = torch.relu(random_layer(x))
show_images([random_maps[0, i] for i in range(8)], [f"Feature map {i}" for i in range(8)])

**Observation:** even random filters produce some edge-like and blur-like maps. During **training**, the network adjusts these numbers so that each filter detects something useful for the task (for example, the edge of a cat's ear).

### 10.3 Stacking layers: how feature maps change with depth

Real CNNs stack several convolution layers. Between them, **max pooling** takes the largest value in each 2 x 2 block, which halves the size. Let us follow the shape through a small network.

In [ ]:
torch.manual_seed(0)

conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
pool = nn.MaxPool2d(kernel_size=2)

print("Input           :", tuple(x.shape))
h = torch.relu(conv1(x));  print("After conv1     :", tuple(h.shape))
h = pool(h);               print("After pooling 1 :", tuple(h.shape))
h = torch.relu(conv2(h));  print("After conv2     :", tuple(h.shape))
h = pool(h);               print("After pooling 2 :", tuple(h.shape))

show_images([h[0, i] for i in range(8)], [f"Layer-2 map {i}" for i in range(8)])

**Observations:**

1. Going deeper, the maps get **smaller** (512 -> 256 -> 128) and there are **more** of them (1 -> 8 -> 16).
2. Deeper maps look less like the original photo. In a trained network, early layers detect simple patterns (edges, colours) and deeper layers combine them into complex shapes (eyes, wheels, faces).
3. Some maps may be almost completely **black**. This is normal: ReLU turns every negative value into 0, and a random filter can give mostly negative outputs. Training fixes this by changing the filter values.

---
## Session 1 Key Takeaways

1. An image is a grid of numbers: `(H, W)` for grayscale, `(H, W, 3)` for colour. PyTorch uses `(N, C, H, W)`.
2. Convolution slides a small kernel over the image, multiplies and adds.
3. The kernel values decide what the output responds to: blur, sharpen, edges.
4. Output size = `(n + 2p - k) // s + 1`.
5. Bigger kernel: more context, smaller output. Bigger stride: much smaller output. Padding: keeps the size.
6. Each filter in a conv layer produces one feature map. In a CNN, the filters are learned.

---
# SESSION 2: Graded Exercises (100 marks)
---

**Instructions**

- Run the setup cells in Session 1 (Part 0, and the `apply_filter` cell in Part 3) before you start. Restarting the notebook? Run them again.
- Write your code where you see `# YOUR CODE HERE` and replace each `...`.
- Write short answers in the cells marked **Your answer**.
- You may look back at Session 1 at any time.

| Exercise | Topic | Marks |
|---|---|---|
| 1 | Output size, padding and stride | 30 |
| 2 | Applying filters and edge detection | 35 |
| 3 | Convolution layers and feature maps | 35 |
| **Total** | | **100** |

In all exercises we use a new image: a photo of coins.

In [ ]:
coins = data.coins()           # grayscale image of coins
xc = to_tensor(coins)          # PyTorch tensor (1, 1, H, W)

print("Coins image shape :", coins.shape)
print("Coins tensor shape:", xc.shape)
show_images([coins], ["Coins"])

## Exercise 1: Output Size, Padding and Stride (30 marks)

### 1(a) Write the output size formula (10 marks)

Complete the function so it returns the output size using

$$\text{output} = \left\lfloor \frac{n + 2p - k}{s} \right\rfloor + 1$$

Hint: use `//` for "round down" division.

In [ ]:
def output_size(n, k, s, p):
    result = (n + 2 * p - k) // s + 1
    return result


print(output_size(32, 3, 1, 0))
print(output_size(32, 3, 1, 1))
print(output_size(32, 5, 2, 2))
print(output_size(28, 7, 3, 0))

### 1(b) Find the "same" padding (10 marks)

With **stride 1**, find the padding that keeps the coins image the **same size** for each kernel size. Set the three variables below, then run the cell. The cell checks your answers using PyTorch.

In [ ]:
p_for_k3 = 1
p_for_k5 = 2
p_for_k7 = 3


H, W = coins.shape
for k, p in [(3, p_for_k3), (5, p_for_k5), (7, p_for_k7)]:
    kernel = np.ones((k, k)) / (k * k)
    out = apply_filter(xc, kernel, stride=1, padding=p)
    print(f"k = {k}, p = {p}: predicted {output_size(H, k, 1, p)} x {output_size(W, k, 1, p)}, "
          f"PyTorch gives {out.shape[2]} x {out.shape[3]}, same size: {out.shape[2:] == xc.shape[2:]}")

### 1(c) Short answers (10 marks)

Write one or two sentences for each.

1. **(5 marks)** An image is 64 x 64. You use a 3 x 3 kernel with padding 1. What is the output size with stride 1, and with stride 2? What happens to the output when you double the stride?
2. **(5 marks)** Why can zero padding create a dark frame around a blurred image, and how can this be avoided?

**Your answer:**

1. With stride 1, the output is 64 x 64. With stride 2, the output is 32 x 32. Doubling the stride makes the output smaller because the kernel moves two pixels at a time.

2. Zero padding adds black pixels around the image, so the blur near the edges can become darker. We can avoid this by using reflection padding instead of zero padding.

## Exercise 2: Applying Filters and Edge Detection (35 marks)

### 2(a) The Laplacian filter (10 marks)

The **Laplacian** kernel finds edges in **all directions** at once:

```
 0   1   0
 1  -4   1
 0   1   0
```

1. Create it as a NumPy array called `laplacian`.
2. Apply it to `xc` with `padding=1` and store the result in `out_lap`.
3. Display the **absolute value** of the result next to the original coins image.

In [ ]:
laplacian = np.array([[0, 1, 0],
                       [1, -4, 1],
                       [0, 1, 0]])

out_lap = apply_filter(xc, laplacian, padding=1)

show_images([coins, out_lap.abs()], ["Original", "Laplacian"])

### 2(b) Edge strength from Sobel X and Sobel Y (15 marks)

Sobel X finds vertical edges and Sobel Y finds horizontal edges. We can combine them into one **edge strength** image:

$$\text{edge} = \sqrt{g_x^2 + g_y^2}$$

1. **(5 marks)** Apply `sobel_x` and `sobel_y` (from Session 1, Part 4) to `xc` with `padding=1`. Store them in `gx` and `gy`.
2. **(5 marks)** Compute `edge = torch.sqrt(gx**2 + gy**2)`.
3. **(5 marks)** Display `gx.abs()`, `gy.abs()` and `edge` side by side with suitable titles.

In [ ]:
gx = apply_filter(xc, sobel_x, padding=1)
gy = apply_filter(xc, sobel_y, padding=1)

edge = torch.sqrt(gx**2 + gy**2)

show_images([gx.abs(), gy.abs(), edge], ["Sobel X", "Sobel Y", "Edge strength"])

### 2(c) Blur first, then find edges (10 marks)

Photos contain small noise and texture. A common trick is to **blur** the image before detecting edges.

1. **(5 marks)** Apply a **5 x 5 box blur** to `xc` (use `padding=2`) and store it in `blurred`. Then compute the edge strength of `blurred` exactly as in 2(b) and store it in `edge_blurred`. Display `edge` and `edge_blurred` side by side.

In [ ]:
blur5 = np.ones((5, 5)) / 25
blurred = apply_filter(xc, blur5, padding=2)

gx_b = apply_filter(blurred, sobel_x, padding=1)
gy_b = apply_filter(blurred, sobel_y, padding=1)
edge_blurred = torch.sqrt(gx_b**2 + gy_b**2)

show_images([edge, edge_blurred], ["Edges (no blur)", "Edges (after 5x5 blur)"])

2. **(5 marks)** In two sentences: how is `edge_blurred` different from `edge`, and why?

**Your answer:**

The blurred edge image has fewer small and noisy details, so the edges look smoother. This happens because the blur averages nearby pixels before the Sobel filters are applied.

## Exercise 3: Convolution Layers and Feature Maps (35 marks)

### 3(a) Build a layer and predict its output (10 marks)

1. **(5 marks)** Create a layer `my_conv` with `nn.Conv2d` that has **1** input channel, **6** output channels, kernel size **5**, stride **2**, padding **2**. (The seed line makes everyone's random filters the same.)
2. **(5 marks)** Use your `output_size` function to **predict** the output height and width for the coins image. Then pass `xc` through the layer and print the real output shape. They should match.

In [ ]:
torch.manual_seed(0)

my_conv = nn.Conv2d(1, 6, kernel_size=5, stride=2, padding=2)

H, W = coins.shape
predicted_h = output_size(H, 5, 2, 2)
predicted_w = output_size(W, 5, 2, 2)
print("Predicted output:", predicted_h, "x", predicted_w)

out = my_conv(xc)
print("Real output shape:", out.shape)

### 3(b) Visualise the feature maps (10 marks)

Apply **ReLU** to `out` (use `torch.relu`) and display all **6** feature maps with titles `"Map 0"`, `"Map 1"`, ... `"Map 5"`.

Hint: look at Part 10.2. The feature map number `i` is `feature_maps[0, i]`.

Note: some of your maps may be almost completely black. This is not a mistake (see Part 10.3, observation 3).

In [ ]:
feature_maps = torch.relu(out)

maps = [feature_maps[0, i] for i in range(6)]
titles = [f"Map {i}" for i in range(6)]

show_images(maps, titles, cols=3)

### 3(c) Kernel size experiment (10 marks)

For kernel sizes **3, 11 and 25**, apply a box blur to `xc` with **stride 1 and padding 0**. For each one, print the output shape and collect the result so all three are displayed side by side. Complete the loop below.

In [ ]:
images = []
titles = []

for k in [3, 11, 25]:
    kernel = np.ones((k, k)) / (k * k)
    result = apply_filter(xc, kernel, stride=1, padding=0)
    print(f"k = {k}: output shape {tuple(result.shape)}")
    images.append(result)
    titles.append(f"k = {k}")

show_images(images, titles)

### 3(d) Short answer (5 marks)

From your results in 3(c): which kernel size gave the **most blur** and the **smallest output**? Explain why in one or two sentences.

**Your answer:**

The 25 x 25 kernel gave the most blur and the smallest output. It averages a larger area of the image, and without padding the output dimensions decrease as the kernel gets bigger.